# 03 — Exercises: cleaning the presales table

**Block 1 — Business & Data Foundations · Lesson 03 · Exercises**

Last week you read SAP, the CRM and the presales spreadsheet into one table of 2,000 bids.
This week the same table comes back the way it really arrives after a few months in
production: bids exported twice, stages typed by hand, dates stored as text, margins
missing, costs nobody could have paid — and the names and e-mails of real people in it.

Your job: **turn it into a table you can trust, without deleting anything silently, and
without keeping personal data you do not need.** The exercises are the closest thing to
your Assignment Brief.

| # | Step | What you practise | Time |
|---|---|---|---|
| 1 | Profile | shape, duplicates, missing share, messy labels | 5 min |
| 2 | Types & categories | text → datetime, a controlled vocabulary | 10 min |
| 3 | Duplicates | one row per bid, keeping the latest export | 5 min |
| 4 | Missing values | fill *and* flag, a third state for labels | 10 min |
| 5 | Outliers | flag impossible values, never delete them | 5 min |
| 6 | Boxplots | outliers per group, not over the whole table | 10 min |
| 7 | Many dimensions | scatter matrix, 3-D plot, Mahalanobis distance | 15 min |
| 8 | Governance + pipeline | pseudonymise, then chain every step | 10 min |

### How to work

- Fill in every cell marked **`# TODO`**. The cell right after it checks your answer and
  prints `✅ Exercise N passed`, or an error message telling you what is still wrong.
- Exercises 2 to 8 ask for a **function** that receives a DataFrame and **returns a new
  one**. Never clean `raw` in place: start every function with `out = df.copy()`.
- **Do not edit the check cells**, and do not edit the cells marked *given*.
- Everything runs **offline**, in memory. Nothing is downloaded or written to disk.
- Stuck? The lesson slides have the matching section, and every exercise names it.

## 0. Setup — *given, run it and move on*

In [1]:
import hashlib
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# Interactive charts in JupyterLab / VS Code, plus an HTML fallback that loads plotly.js from
# a CDN (keeps the saved notebook small). In Google Colab use: pio.renderers.default = "colab"
pio.renderers.default = "plotly_mimetype+notebook_connected"
pio.templates.default = "plotly_white"

SEED = 7
rng = np.random.default_rng(SEED)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)

# The secret for exercise 8. In a real project it lives in a secrets vault, never in a notebook.
SALT = "msmk-presales-2026"


def passed(n):
    print(f"✅ Exercise {n} passed")


print("pandas", pd.__version__)

pandas 3.0.6


## 1. Last week's table, as it really arrives — *given*

The cell below rebuilds the lesson 02 analysis table and then does to it what months of
production do: a re-export, hand-typed labels, typos, gaps. **Do not read the cell for the
answers** — find the problems the way you would in a real table, in exercise 1.

**In your Assignment Brief this cell disappears** and you start from your own joined table.

In [2]:
# ---------------------------------------------------------------- the bid pipeline (lesson 02)
N_BIDS = 2_000
SECTORS = ["Naval", "Energía", "Defensa", "Logística", "Infraestructura"]
STAGES = ["Elaboración", "Entregada", "Ganada", "Perdida"]
ACCOUNTS = [f"ACC{i:04d}" for i in range(120)]
OWNERS = [f"OWN{i:03d}" for i in range(25)]
start = datetime(2025, 9, 1)

bids = pd.DataFrame({
    "bid_id": [f"BID{i:05d}" for i in range(N_BIDS)],
    "account_id": rng.choice(ACCOUNTS, N_BIDS),
    "owner_id": rng.choice(OWNERS, N_BIDS),
    "opened_at": [start + timedelta(days=int(d)) for d in np.sort(rng.integers(0, 365, N_BIDS))],
    "sector": rng.choice(SECTORS, N_BIDS),
    "cost_eur": np.round(rng.gamma(3.0, 42_000, N_BIDS), 2),
    "margin_pct": np.round(np.clip(rng.normal(18, 6, N_BIDS), -5, 45), 2),
    "stage": rng.choice(STAGES, N_BIDS, p=[.18, .22, .29, .31]),
})
bids.loc[rng.random(N_BIDS) < 0.06, "margin_pct"] = np.nan

# ---------------------------------------------------------------- the columns the joins added
acc = bids["account_id"].str[3:].astype(int)
own = bids["owner_id"].str[3:].astype(int)
bids["account_name"] = [f"Cliente {i:03d}, S.A." for i in acc]
bids["country"] = rng.choice(["ES", "PT", "FR", "IT", "MA"], 120, p=[.62, .13, .11, .09, .05])[acc]
bids["contact_email"] = [f"contacto{i:03d}@cliente{i:03d}.com" for i in acc]
bids["owner_name"] = [f"Comercial {i:02d}" for i in own]
bids["owner_email"] = [f"comercial{i:02d}@msmk-consulting.es" for i in own]
qualification = np.full(N_BIDS, None, dtype=object)
qualification[rng.choice(N_BIDS, 1_400, replace=False)] = rng.choice(
    ["estrategica", "tactica", "oportunista"], 1_400)
bids["qualification"] = pd.Series(qualification, dtype="str")

# ---------------------------------------------------------------- months of production
SPELLINGS = {                                        # three people, two years, one free-text box
    "Elaboración": ["Elaboración", "elaboracion", "ELABORACIÓN ", "Elaboracion"],
    "Entregada": ["Entregada", "entregada ", "ENTREGADA"],
    "Ganada": ["Ganada", "ganada", " GANADA", "Ganada "],
    "Perdida": ["Perdida", "perdida ", "PERDIDA"],
}
bids["stage"] = [str(rng.choice(SPELLINGS[s])) for s in bids["stage"]]

typos = rng.choice(bids.index, 6, replace=False)     # a minus sign, a zero, three extra zeros
bids.loc[typos[:3], "cost_eur"] = [-1_850.0, 0.0, -42_300.0]
bids.loc[typos[3:], "cost_eur"] *= 1_000

closed = bids.index[bids["stage"].str.strip().str.casefold().isin(["ganada", "perdida"])]
again = np.sort(rng.choice(closed.difference(typos), 40, replace=False))
latest = bids.loc[again].copy()                      # the re-export knows how the bid ended ...
bids.loc[again, "stage"] = "Entregada"               # ... the first export did not

raw = pd.concat([bids, latest], ignore_index=True)
raw["opened_at"] = raw["opened_at"].dt.strftime("%d/%m/%Y")   # somebody saved it as text
del bids, latest

print(raw.shape)
raw.head()

(2040, 14)


,bid_id,account_id,owner_id,opened_at,sector,cost_eur,margin_pct,stage,account_name,country,contact_email,owner_name,owner_email,qualification
0,BID00000,ACC0113,OWN015,01/09/2025,Logística,57700.40,15.83,Perdida,"Cliente 113, S.A.",ES,contacto113@cliente113.com,Comercial 15,comercial15@msmk-consulting.es,tactica
1,BID00001,ACC0075,OWN021,01/09/2025,Energía,142236.57,17.31,Ganada,"Cliente 075, S.A.",MA,contacto075@cliente075.com,Comercial 21,comercial21@msmk-consulting.es,NaN
2,BID00002,ACC0082,OWN006,01/09/2025,Energía,84481.86,NaN,Elaboracion,"Cliente 082, S.A.",IT,contacto082@cliente082.com,Comercial 06,comercial06@msmk-consulting.es,oportunista
3,BID00003,ACC0107,OWN021,01/09/2025,Infraestructura,64714.85,16.90,Ganada,"Cliente 107, S.A.",MA,contacto107@cliente107.com,Comercial 21,comercial21@msmk-consulting.es,NaN
4,BID00004,ACC0069,OWN001,01/09/2025,Naval,71473.48,16.68,ganada,"Cliente 069, S.A.",ES,contacto069@cliente069.com,Comercial 01,comercial01@msmk-consulting.es,NaN


---

## Exercise 1 — Profile before you touch it

*Lesson section 01 · Profile first · 5 minutes*

Never clean a table you have not measured. Compute four numbers from `raw`, and **do not
change `raw`**:

| Name | What it holds |
|---|---|
| `n_rows` | the number of rows |
| `n_dup_bids` | how many rows repeat a `bid_id` that already appeared above them |
| `missing_share` | a Series: the share of missing values in each column (0 to 1) |
| `n_stage_labels` | how many different ways the `stage` column is written |

Then look at your numbers: which columns have gaps, and how bad is `stage`?

> **Hint.** `len`, `.duplicated(...).sum()`, `.isna().mean()` and `.nunique()`.
> `raw["stage"].value_counts()` shows you the spellings.

In [3]:
# TODO: profile `raw` — four numbers, no changes to the table.
n_rows = len(raw)
n_dup_bids = raw["bid_id"].duplicated().sum()
missing_share = raw.isna().mean()
n_stage_labels = raw["stage"].nunique()

print(f"Filas totales: {n_rows}")
print(f"Bids duplicadas: {n_dup_bids}")
print(f"Etiquetas de stage: {n_stage_labels}")
missing_share

Filas totales: 2040
Bids duplicadas: 40
Etiquetas de stage: 14


bid_id           0.000000
account_id       0.000000
owner_id         0.000000
opened_at        0.000000
sector           0.000000
cost_eur         0.000000
margin_pct       0.061765
stage            0.000000
account_name     0.000000
country          0.000000
contact_email    0.000000
owner_name       0.000000
owner_email      0.000000
qualification    0.300490
dtype: float64

In [4]:
# --- check 1 (do not edit) ---
assert n_rows == 2_040, f"n_rows should count every row of raw, got {n_rows}"
assert n_dup_bids == 40, \
    f"expected 40 repeated bids, got {n_dup_bids} — use raw['bid_id'].duplicated().sum()"
assert isinstance(missing_share, pd.Series), "missing_share must be a Series — use raw.isna().mean()"
assert set(raw.columns) <= set(missing_share.index), "missing_share needs one value per column of raw"
assert abs(missing_share["margin_pct"] - 126 / 2_040) < 1e-9, \
    "missing_share must be a share between 0 and 1 — mean(), not sum()"
assert abs(missing_share["qualification"] - 613 / 2_040) < 1e-9, "the qualification share is wrong"
assert n_stage_labels == 14, \
    f"expected 14 ways of writing stage, got {n_stage_labels} — count them on raw, before cleaning"
assert len(raw) == 2_040, "raw was changed — profiling must not modify the table"
passed(1)

✅ Exercise 1 passed


---

## Exercise 2 — Fix the types and the categories

*Lesson section 02 · Types, dates and categories · 10 minutes*

Write a function `fix_types(df)` that returns a **new** DataFrame where:

- `opened_at` is a real **datetime**, parsed with the format it was written in
  (`dd/mm/yyyy`) — the earliest bid is in **September 2025**;
- `stage` holds exactly **four** values — `elaboracion`, `entregada`, `ganada`,
  `perdida` — with the `category` dtype. Get there in this order: remove the surrounding
  spaces, `casefold`, remove the accents.

Every other column stays as it is, and so does the number of rows.

> **Hint.** `pd.to_datetime(out["opened_at"], format="%d/%m/%Y")`. For the accents:
> `.str.normalize("NFKD").str.encode("ascii", "ignore").str.decode("ascii")` splits `ó`
> into `o` + an accent mark and then drops the mark. Finish with `.astype("category")`.

In [5]:
def fix_types(df):
    out = df.copy()
    out["opened_at"] = pd.to_datetime(out["opened_at"], format="%d/%m/%Y")
    out["stage"] = (
        out["stage"]
        .str.strip()
        .str.casefold()
        .str.normalize("NFKD")
        .str.encode("ascii", "ignore")
        .str.decode("ascii")
        .astype("category")
    )
    
    return out

df_fixed = fix_types(raw)

print("--- Tipos de datos ---")
print(df_fixed[["opened_at", "stage"]].dtypes)

print("\n--- Categorías limpias de 'stage' ---")
print(df_fixed["stage"].value_counts())

--- Tipos de datos ---
opened_at    datetime64[us]
stage              category
dtype: object

--- Categorías limpias de 'stage' ---
stage
perdida        632
ganada         564
entregada      482
elaboracion    362
Name: count, dtype: int64


In [6]:
# --- check 2 (do not edit) ---
from pandas.api.types import is_datetime64_any_dtype

inp = raw.copy()
out = fix_types(inp)
assert isinstance(out, pd.DataFrame), "fix_types must return a DataFrame — replace the `...` with `out`"
assert inp.equals(raw), "fix_types changed its input — start with `out = df.copy()`"
assert len(out) == 2_040, f"fix_types must keep every row, got {len(out)}"
assert is_datetime64_any_dtype(out["opened_at"]), "opened_at is not a datetime — use pd.to_datetime"
assert (out["opened_at"].min().year, out["opened_at"].min().month) == (2025, 9), \
    "the earliest bid is not in September 2025 — give pd.to_datetime format='%d/%m/%Y'"
assert isinstance(out["stage"].dtype, pd.CategoricalDtype), "stage must have the category dtype"
assert set(out["stage"]) == {"elaboracion", "entregada", "ganada", "perdida"}, \
    f"expected four tidy stages, got {sorted(set(out['stage']))} — strip, casefold, remove accents"
passed(2)

✅ Exercise 2 passed


---

## Exercise 3 — One row per bid

*Lesson section 02 · Duplicates · 5 minutes*

SAP re-exported 40 bids after they closed. The first copy still says `Entregada`; the
**later** copy — further down the table — knows whether the bid was won or lost.

Write `drop_duplicate_bids(df)` returning a new DataFrame with **one row per `bid_id`**,
keeping the **latest** export of each bid.

> **Hint.** `drop_duplicates` takes `subset=` (which columns define "the same bid") and
> `keep=` (which copy survives).

In [7]:
# TODO: complete the function.
def drop_duplicate_bids(df):
    out = df.copy()
    out = out.drop_duplicates(subset=["bid_id"], keep="last")
    return out

df_dedup = drop_duplicate_bids(raw)
print(f"Filas originales: {len(raw)}")
print(f"Filas tras eliminar duplicados: {len(df_dedup)}")

Filas originales: 2040
Filas tras eliminar duplicados: 2000


In [8]:
# --- check 3 (do not edit) ---
inp = raw.copy()
out = drop_duplicate_bids(inp)
assert isinstance(out, pd.DataFrame), "drop_duplicate_bids must return a DataFrame — replace the `...`"
assert inp.equals(raw), "drop_duplicate_bids changed its input — start with `out = df.copy()`"
assert out["bid_id"].is_unique, "bid_id is still repeated — use drop_duplicates(subset=['bid_id'])"
assert len(out) == 2_000, f"expected 2000 bids, got {len(out)} — deduplicate on bid_id only"
again = raw.loc[raw["bid_id"].duplicated(), "bid_id"]
kept = out.loc[out["bid_id"].isin(again), "stage"].str.strip().str.casefold()
assert not kept.eq("entregada").any(), \
    "you kept the first, outdated export of the re-exported bids — use keep='last'"
passed(3)

✅ Exercise 3 passed


---

## Exercise 4 — Fill, and say that you filled

*Lesson section 02 · Missing values · 10 minutes*

Write `handle_missing(df)` returning a new DataFrame where:

1. a new column **`margin_pct_missing`** is `True` where `margin_pct` was missing — create
   it **before** you fill anything;
2. the missing `margin_pct` values are filled with the **median margin of the bid's
   sector** (not the overall median);
3. the missing `qualification` values become the string **`"unknown"`** — the bid stays
   in the table, as a third state next to the three real qualifications.

No row is dropped.

> **Hint.** `out["margin_pct"].isna()` is the flag.
> `out.groupby("sector")["margin_pct"].transform("median")` gives every row its sector's
> median, lined up with the table; pass it to `.fillna(...)`.

In [9]:
# TODO: complete the function.
def handle_missing(df):
    out = df.copy()
    
    # 1. Crear el indicador ANTES de rellenar los valores nulos
    out["margin_pct_missing"] = out["margin_pct"].isna()
    
    # 2. Rellenar los nulos de margin_pct con la mediana de cada sector
    sector_medians = out.groupby("sector")["margin_pct"].transform("median")
    out["margin_pct"] = out["margin_pct"].fillna(sector_medians)
    
    # 3. Rellenar los nulos de qualification con el texto "unknown"
    out["qualification"] = out["qualification"].fillna("unknown")
    
    return out

# Guardamos el resultado en la variable antes de imprimirla
resultado = handle_missing(raw)[["margin_pct", "margin_pct_missing", "qualification"]].head()
print(resultado)

   margin_pct  margin_pct_missing qualification
0       15.83               False       tactica
1       17.31               False       unknown
2       18.62                True   oportunista
3       16.90               False       unknown
4       16.68               False       unknown


In [10]:
# --- check 4 (do not edit) ---
inp = raw.copy()
out = handle_missing(inp)
assert isinstance(out, pd.DataFrame), "handle_missing must return a DataFrame — replace the `...`"
assert inp.equals(raw), "handle_missing changed its input — start with `out = df.copy()`"
assert len(out) == 2_040, f"handle_missing must not drop rows, got {len(out)}"
assert "margin_pct_missing" in out.columns, "add the flag column margin_pct_missing"
assert out["margin_pct_missing"].dtype == bool and out["margin_pct_missing"].sum() == 126, \
    "margin_pct_missing must be True exactly where margin_pct was missing — create it before filling"
assert out["margin_pct"].notna().all(), "margin_pct still has missing values"
flag = out["margin_pct_missing"]
medians = raw.groupby("sector")["margin_pct"].transform("median")
assert np.allclose(out.loc[flag, "margin_pct"], medians[flag]), \
    "the filled margins are not the sector medians — use groupby('sector') with transform('median')"
assert out.loc[~flag, "margin_pct"].equals(raw.loc[~flag, "margin_pct"]), "known margins must not change"
assert (out["qualification"] == "unknown").sum() == 613 and out["qualification"].notna().all(), \
    "the 613 unqualified bids must say 'unknown'"
passed(4)

✅ Exercise 4 passed


---

## Exercise 5 — Flag the impossible, delete nothing

*Lesson section 02 · Outliers · 5 minutes*

Bid costs are skewed — a few big defence tenders cost many times the typical bid, and
they are real. First look at what a purely statistical rule would do (given):

In [11]:
q1, q3 = raw["cost_eur"].quantile([0.25, 0.75])
fence = q3 + 1.5 * (q3 - q1)
print(f"IQR fence: {fence:,.0f} EUR -> {(raw['cost_eur'] > fence).sum()} bids above it")
raw["cost_eur"].describe()

IQR fence: 306,659 EUR -> 51 bids above it


count    2.040000e+03
mean     2.447277e+05
std      3.233723e+06
min     -4.230000e+04
25%      7.111703e+04
50%      1.130436e+05
75%      1.653337e+05
max      1.096738e+08
Name: cost_eur, dtype: float64

Most of those bids are simply large. The domain rule is sharper: **a bid cost is
impossible if it is ≤ 0 or above 5,000,000 EUR** (nobody spends five million preparing an
offer).

Write `flag_outliers(df)` returning a new DataFrame with a boolean column **`cost_outlier`**
that is `True` where that rule says the cost is impossible. Keep every row, and leave
`cost_eur` untouched.

> **Hint.** `(out["cost_eur"] <= 0) | (out["cost_eur"] > 5_000_000)` — the brackets matter.

In [12]:
# TODO: complete the function.
def flag_outliers(df):
    out = df.copy()
    
    # Creamos la columna booleana cost_outlier según la regla del negocio
    out["cost_outlier"] = (out["cost_eur"] <= 0) | (out["cost_eur"] > 5_000_000)
    
    return out

# Probamos la función y mostramos las filas marcadas como outliers
flagged = flag_outliers(raw)
flagged.loc[flagged["cost_outlier"], ["bid_id", "cost_eur"]]

,bid_id,cost_eur
143,BID00143,79163150.0
154,BID00154,109673820.0
614,BID00614,0.0
969,BID00969,-1850.0
1343,BID01343,-42300.0
1861,BID01861,55735520.0


In [13]:
# --- check 5 (do not edit) ---
inp = raw.copy()
out = flag_outliers(inp)
assert isinstance(out, pd.DataFrame), "flag_outliers must return a DataFrame — replace the `...`"
assert inp.equals(raw), "flag_outliers changed its input — start with `out = df.copy()`"
assert len(out) == 2_040, f"flag, do not delete: expected 2040 rows, got {len(out)}"
assert "cost_outlier" in out.columns and out["cost_outlier"].dtype == bool, \
    "add a boolean column cost_outlier"
assert out["cost_eur"].equals(raw["cost_eur"]), "cost_eur must stay untouched — flag it, do not fix it"
impossible = (raw["cost_eur"] <= 0) | (raw["cost_eur"] > 5_000_000)
assert out["cost_outlier"].equals(impossible), \
    f"{out['cost_outlier'].sum()} bids flagged, expected {impossible.sum()} — cost <= 0 or > 5,000,000"
passed(5)

✅ Exercise 5 passed


---

## Exercise 6 — Boxplots: an outlier for whom?

*Lesson section 02 · Outliers · 10 minutes*

The costs of exercise 5 came from SAP. Presales also keeps its own record of the **effort**
behind each bid: hours spent, people on the team, pages of the offer. The cell below adds
that table — one row per bid, the 2,000 bids of `raw`. Again, **do not read it for the
answers.**

In [14]:
# ---------------------------------------------------------------- the presales effort log
erng = np.random.default_rng(SEED + 1)               # its own generator: raw stays as it was
effort = raw.drop_duplicates("bid_id")[["bid_id", "sector"]].reset_index(drop=True)
n = len(effort)
SCALE = {"Defensa": 2.0, "Naval": 1.5, "Energía": 1.2, "Infraestructura": 1.0, "Logística": 0.5}
size = erng.normal(0, 1, n)                          # how big the bid is, for its sector
hours = effort["sector"].map(SCALE).to_numpy() * (160 + 45 * size + erng.normal(0, 12, n))
effort["hours"] = np.round(np.clip(hours, 8, None), 1)
effort["pages"] = np.clip(np.round(effort["hours"] * 0.6 + erng.normal(0, 8, n)), 5, None).astype(int)
effort["team_size"] = np.clip(np.round(1.5 + effort["hours"] / 70 + erng.normal(0, .6, n)), 1, None).astype(int)

# ---------------------------------------------------------------- what the log also holds
big = erng.choice(effort.index[effort["sector"] == "Logística"], 5, replace=False)
effort.loc[big, "hours"] = np.round(erng.uniform(190, 230, 5), 1)
effort.loc[big, "pages"] = np.round(effort.loc[big, "hours"] * 0.6).astype(int)
effort.loc[big, "team_size"] = np.round(1.5 + effort.loc[big, "hours"] / 70).astype(int)
busy = effort.index.difference(big)[effort.drop(big)["hours"].between(230, 300).to_numpy()]
odd = erng.choice(busy, 6, replace=False)
effort.loc[odd, "pages"] = erng.integers(15, 35, 6)
effort.loc[odd, "team_size"] = 1
del erng, n, size, hours, big, busy, odd

print(effort.shape)
effort.head()

(2000, 5)


,bid_id,sector,hours,pages,team_size
0,BID00000,Logística,41.0,25,1
1,BID00001,Energía,127.0,74,4
2,BID00002,Energía,147.9,85,4
3,BID00003,Infraestructura,160.2,102,4
4,BID00004,Naval,74.2,40,3


A boxplot draws the same rule as the IQR fence in exercise 5: the box goes from Q1 to Q3,
the whiskers stop at the last value inside **Q1 − 1.5·IQR** and **Q3 + 1.5·IQR**, and every
value beyond them is drawn as a separate dot — a candidate outlier. One box over all the bids
(given):

In [15]:
px.box(effort, y="hours", points="outliers", title="Hours per bid — every sector in one box")

The dots are mostly **Defensa** bids — the sector that always takes longest. They are not
wrong; they are big. Is 200 hours a lot? It depends on *whose* bid it is. Compare every bid
with **its own sector**:

1. **`fig_box`** — a plotly boxplot of `hours` with **one box per `sector`**, drawing the
   outliers as dots.
2. **`flag_sector_outliers(df)`** — returns a new DataFrame with a boolean column
   **`hours_outlier`** that is `True` where `hours` is below `Q1 − 1.5·IQR` or above
   `Q3 + 1.5·IQR` **of the bid's sector** (use pandas' default `quantile`). Keep every row.

Then look at the boxes: which sector has dots that the single box above did not show?

> **Hint.** `px.box(effort, x="sector", y="hours", points="outliers")`.
> `out.groupby("sector")["hours"].transform(lambda s: s.quantile(0.25))` gives every row its
> sector's Q1, lined up with the table — the same trick as the sector median in exercise 4.

In [16]:
# TODO: one box per sector, and a flag against the bid's own sector.
fig_box = px.box(effort, x="sector", y="hours", points="outliers")
# fig_box.show()   # Descomenta esta línea si quieres mostrar la gráfica


def flag_sector_outliers(df):
    out = df.copy()
    
    # 1. Calculamos Q1, Q3 e IQR por sector usando groupby y transform
    g = out.groupby("sector")["hours"]
    q1 = g.transform(lambda s: s.quantile(0.25))
    q3 = g.transform(lambda s: s.quantile(0.75))
    iqr = q3 - q1
    
    # 2. Marcamos las filas cuyo valor está fuera de los límites (fences) de su propio sector
    out["hours_outlier"] = (out["hours"] < q1 - 1.5 * iqr) | (out["hours"] > q3 + 1.5 * iqr)
    
    return out


flagged = flag_sector_outliers(effort)
flagged.loc[flagged["hours_outlier"]].sort_values(["sector", "hours"])

,bid_id,sector,hours,pages,team_size,hours_outlier
797,BID00797,Defensa,21.4,20,2,True
1504,BID01504,Defensa,40.9,21,2,True
220,BID00220,Defensa,53.1,31,3,True
174,BID00174,Energía,8.0,7,2,True
713,BID00713,Energía,353.9,214,7,True
413,BID00413,Infraestructura,31.9,18,2,True
1488,BID01488,Logística,13.3,11,2,True
845,BID00845,Logística,143.1,87,4,True
762,BID00762,Logística,155.1,92,3,True
1183,BID01183,Logística,191.5,115,4,True


In [17]:
# --- check 6 (do not edit) ---
assert isinstance(fig_box, go.Figure) and fig_box.data, "fig_box must be a plotly figure — use px.box"
assert all(t.type == "box" for t in fig_box.data), "fig_box must be a boxplot — use px.box"
xs = set().union(*(set(t.x) for t in fig_box.data if t.x is not None))
assert xs == set(effort["sector"]), "fig_box needs one box per sector — pass x='sector'"
assert all(t.boxpoints in (None, "outliers", "suspectedoutliers", "all") for t in fig_box.data)

inp = effort.copy()
out = flag_sector_outliers(inp)
assert isinstance(out, pd.DataFrame), "flag_sector_outliers must return a DataFrame — replace the `...`"
assert inp.equals(effort), "flag_sector_outliers changed its input — start with `out = df.copy()`"
assert len(out) == len(effort), "flag, do not delete: every bid must stay"
assert "hours_outlier" in out.columns and out["hours_outlier"].dtype == bool, \
    "add a boolean column hours_outlier"
g = effort.groupby("sector")["hours"]
q1, q3 = g.transform(lambda s: s.quantile(0.25)), g.transform(lambda s: s.quantile(0.75))
expected = (effort["hours"] < q1 - 1.5 * (q3 - q1)) | (effort["hours"] > q3 + 1.5 * (q3 - q1))
Q1, Q3 = effort["hours"].quantile([0.25, 0.75])
overall = (effort["hours"] < Q1 - 1.5 * (Q3 - Q1)) | (effort["hours"] > Q3 + 1.5 * (Q3 - Q1))
assert not out["hours_outlier"].equals(overall), \
    "these are the fences of the whole table — compute Q1 and Q3 per sector with groupby + transform"
assert out["hours_outlier"].equals(expected), \
    f"{out['hours_outlier'].sum()} bids flagged, expected {expected.sum()} — Q1 - 1.5·IQR / Q3 + 1.5·IQR per sector"
passed(6)

✅ Exercise 6 passed


---

## Exercise 7 — Outliers you only see in several dimensions

*Lesson section 02 · Outliers · 15 minutes*

A boxplot looks at **one column at a time**. Some bids break no single-column rule and are
still impossible: 260 hours of work — normal — delivered by *one person* — possible — as a
*20-page* offer — also possible. Together, they do not add up. The first step is to **see**
several dimensions at once (given): each small panel below is the scatter plot of one pair
of columns.

In [25]:
EFFORT = ["hours", "team_size", "pages"]
px.scatter_matrix(effort, dimensions=EFFORT, opacity=0.4, height=650,
                  title="Effort per bid — every pair of columns")

Hours, pages and team size grow **together**: the points lie along a band. A few points sit
well off the band in the `pages`–`hours` panel, although neither of their values is extreme
on its own.

The **Mahalanobis distance** turns "off the band" into a number. It measures how far a row
is from the average row, in units of the data's own spread, **taking the correlations into
account**: a step along the band is cheap, a step across it is expensive.

$$
d(x) = \sqrt{(x - \mu)^\top \, \Sigma^{-1} \, (x - \mu)}
$$

where $\mu$ is the vector of column means and $\Sigma$ the covariance matrix of the columns.
For well-behaved data, $d^2$ follows a $\chi^2$ distribution with one degree of freedom per
column, which gives a threshold instead of a guess (given below).

1. **`mahalanobis(df, cols)`** — returns a **Series** (same index as `df`) with the
   distance of every row, computed over the columns `cols`.
2. **`flag_effort_outliers(df)`** — returns a new DataFrame with two new columns:
   **`effort_distance`** (the distance over `EFFORT`) and **`effort_outlier`**, `True` where
   the distance is above `THRESHOLD`.
3. **`fig_3d`** — a plotly **3-D scatter** of the three `EFFORT` columns of the flagged
   table, coloured by `effort_outlier`. Rotate it: most flagged bids float away from the
   cloud. Are they all equally far?

> **Hint.** `X = df[cols].to_numpy(dtype=float)`, `diff = X - X.mean(axis=0)`,
> `inv = np.linalg.inv(np.cov(X, rowvar=False))`. The squared distance of every row at once is
> `np.einsum("ij,jk,ik->i", diff, inv, diff)` — or `((diff @ inv) * diff).sum(axis=1)`.
> For the plot: `px.scatter_3d(df, x=..., y=..., z=..., color=...)`.

In [19]:
from scipy.stats import chi2

THRESHOLD = np.sqrt(chi2.ppf(0.999, df=len(EFFORT)))  # 1 in 1,000 normal bids would pass it
print(f"threshold: distance > {THRESHOLD:.2f}")

threshold: distance > 4.03


In [26]:
# TODO: the distance, the flag, and a 3-D look at them.
def mahalanobis(df, cols):
    # 1. Convertimos las columnas a un array numpy y restamos la media de cada columna
    X = df[cols].to_numpy(dtype=float)
    diff = X - X.mean(axis=0)
    
    # 2. Calculamos la matriz de covarianza y su inversa
    cov = np.cov(X, rowvar=False)
    inv_cov = np.linalg.inv(cov)
    
    # 3. Calculamos la distancia de Mahalanobis al cuadrado para cada fila
    dist_sq = ((diff @ inv_cov) * diff).sum(axis=1)
    
    # Devuelve una Serie con el mismo índice del DataFrame original
    return pd.Series(np.sqrt(dist_sq), index=df.index)


def flag_effort_outliers(df):
    out = df.copy()
    
    # Añadimos la distancia calculada y el indicador booleano según el umbral THRESHOLD
    out["effort_distance"] = mahalanobis(out, EFFORT)
    out["effort_outlier"] = out["effort_distance"] > THRESHOLD
    
    return out


# Generamos el DataFrame marcado
effort_flagged = flag_effort_outliers(effort)

# Gráfico 3D con Plotly asignando color por la columna effort_outlier
fig_3d = px.scatter_3d(
    effort_flagged, 
    x="hours", 
    y="team_size", 
    z="pages", 
    color="effort_outlier"
)
# fig_3d.show()   # Descomenta para ver el gráfico interactivo 3D

effort_flagged.loc[effort_flagged["effort_outlier"]].sort_values("effort_distance", ascending=False)

,bid_id,sector,hours,pages,team_size,effort_distance,effort_outlier
1835,BID01835,Defensa,291.8,21,1,14.608967,True
1721,BID01721,Energía,296.1,29,1,14.186773,True
1694,BID01694,Naval,280.8,23,1,13.827856,True
209,BID00209,Energía,284.2,31,1,13.359001,True
343,BID00343,Energía,241.8,24,1,11.572794,True
1223,BID01223,Energía,232.3,22,1,11.209211,True
607,BID00607,Defensa,402.9,252,5,4.117000,True


In [27]:
# --- check 7 (do not edit) ---
def _reference(cols):
    X = effort[cols].to_numpy(dtype=float)
    diff = X - X.mean(axis=0)
    return np.sqrt(((diff @ np.linalg.inv(np.cov(X, rowvar=False))) * diff).sum(axis=1))


ref = _reference(EFFORT)
Z = effort[EFFORT].to_numpy(dtype=float)
euclid = np.sqrt((((Z - Z.mean(axis=0)) / Z.std(axis=0, ddof=1)) ** 2).sum(axis=1))

d = mahalanobis(effort, EFFORT)
assert isinstance(d, pd.Series), "mahalanobis must return a Series — wrap the result in pd.Series"
assert d.index.equals(effort.index), "the Series needs the same index as df — pass index=df.index"
assert not np.allclose(d, euclid), \
    "that is a distance on standardised columns — it ignores the correlations; use the inverse covariance"
assert np.allclose(d, ref), \
    "the distances are wrong — sqrt of (x - mu)' inv(cov) (x - mu), with cov = np.cov(X, rowvar=False)"
assert np.allclose(mahalanobis(effort, ["hours", "pages"]), _reference(["hours", "pages"])), \
    "mahalanobis must use the columns it is given in `cols`, not a fixed list"

inp = effort.copy()
out = flag_effort_outliers(inp)
assert isinstance(out, pd.DataFrame), "flag_effort_outliers must return a DataFrame — replace the `...`"
assert inp.equals(effort), "flag_effort_outliers changed its input — start with `out = df.copy()`"
assert len(out) == len(effort), "flag, do not delete: every bid must stay"
assert {"effort_distance", "effort_outlier"} <= set(out.columns), "add effort_distance and effort_outlier"
assert np.allclose(out["effort_distance"], ref), "effort_distance must be mahalanobis(df, EFFORT)"
assert out["effort_outlier"].dtype == bool and out["effort_outlier"].equals(pd.Series(ref > THRESHOLD, index=effort.index)), \
    "effort_outlier must be True exactly where effort_distance > THRESHOLD"
assert out["effort_outlier"].sum() == 7, f"expected 7 bids above the threshold, got {out['effort_outlier'].sum()}"

assert isinstance(fig_3d, go.Figure) and fig_3d.data, "fig_3d must be a plotly figure — use px.scatter_3d"
assert all(t.type == "scatter3d" for t in fig_3d.data), "fig_3d must be a 3-D scatter — use px.scatter_3d"
assert len(fig_3d.data) >= 2, "colour fig_3d by effort_outlier, so the flagged bids stand out"
passed(7)

✅ Exercise 7 passed


Would the boxplots of exercise 6 have found them? Run the per-column fence on each
dimension (given):

In [28]:
flags = effort_flagged[["bid_id", "effort_outlier"]].copy()
for col in EFFORT:
    q1, q3 = effort[col].quantile([0.25, 0.75])
    fence = 1.5 * (q3 - q1)
    flags[f"{col}_box"] = (effort[col] < q1 - fence) | (effort[col] > q3 + fence)
flags["any_box"] = flags[[f"{c}_box" for c in EFFORT]].any(axis=1)
pd.crosstab(flags["effort_outlier"], flags["any_box"],
            rownames=["Mahalanobis outlier"], colnames=["caught by a boxplot"])

caught by a boxplot,False,True
Mahalanobis outlier,,
False,1964,29
True,7,0


---

## Exercise 8 — Pseudonymise, then run the whole pipeline

*Lesson sections 03 (Personal data, pseudonymisation) and 02 (The cleaning log) · 10 minutes*

**8a.** The table carries personal data it does not need to answer "where does presales
time go?". Write `pseudonymise(df)` returning a new DataFrame that:

- **drops** `owner_name` and `contact_email` — nobody needs them for the analysis
  (data minimisation);
- **replaces** `owner_email` with `owner_key`: the SHA-256 hex digest of `SALT + email`.
  The same person always gets the same key, so you can still count bids per owner, but
  the key alone does not say who they are. `owner_email` must not stay in the table.

**8b.** Chain your five functions, in the order given in `STEPS`, into one cleaning run:

- `clean` — the result of passing `raw` through every step;
- `log` — a DataFrame with columns `step` and `rows`: one line per step (a first line for
  `raw` is welcome), so anyone can see where rows disappeared;
- finish with `assert clean["bid_id"].is_unique` — the pipeline must fail loudly if a
  step ever breaks the key.

> **Hint.** For one e-mail:
> `hashlib.sha256((SALT + email).encode("utf-8")).hexdigest()`; apply it to the column
> with `.map(...)`. For 8b, a `for step in STEPS:` loop with `df.pipe(step)` and
> `step.__name__` does both jobs at once.

In [29]:
# TODO (8a): complete the function.
def pseudonymise(df):
    out = df.copy()
    
    # 1. Crear owner_key aplicando SHA-256 sobre SALT + owner_email
    out["owner_key"] = out["owner_email"].map(
        lambda e: hashlib.sha256((SALT + e).encode("utf-8")).hexdigest()
    )
    
    # 2. Eliminar las columnas con datos personales innecesarios
    out = out.drop(columns=["owner_name", "contact_email", "owner_email"])
    
    return out


# TODO (8b): run the pipeline and keep a log.
STEPS = [fix_types, drop_duplicate_bids, handle_missing, flag_outliers, pseudonymise]

# Ejecutamos el pipeline registrando los pasos
current_df = raw.copy()
log_records = [{"step": "raw", "rows": len(current_df)}]

for step in STEPS:
    current_df = step(current_df)
    log_records.append({"step": step.__name__, "rows": len(current_df)})

clean = current_df
log = pd.DataFrame(log_records)

# Verificación de clave única
assert clean["bid_id"].is_unique

log

,step,rows
0,raw,2040
1,fix_types,2040
2,drop_duplicate_bids,2000
3,handle_missing,2000
4,flag_outliers,2000
5,pseudonymise,2000


In [30]:
# --- check 8 (do not edit) ---
from pandas.api.types import is_datetime64_any_dtype

inp = raw.copy()
out = pseudonymise(inp)
assert isinstance(out, pd.DataFrame), "pseudonymise must return a DataFrame — replace the `...`"
assert inp.equals(raw), "pseudonymise changed its input — start with `out = df.copy()`"
for col in ["owner_name", "contact_email", "owner_email"]:
    assert col not in out.columns, f"{col} is still in the table — drop it"
assert "owner_key" in out.columns, "add the owner_key column"
expected = raw["owner_email"].map(lambda e: hashlib.sha256((SALT + e).encode("utf-8")).hexdigest())
assert list(out["owner_key"]) == list(expected), \
    "owner_key must be hashlib.sha256((SALT + email).encode('utf-8')).hexdigest()"
assert out["owner_key"].nunique() == 25, "one key per owner: expected 25 different keys"

assert isinstance(clean, pd.DataFrame), "clean must be the DataFrame the pipeline returns"
assert len(clean) == 2_000 and clean["bid_id"].is_unique, "clean must hold 2000 unique bids"
assert is_datetime64_any_dtype(clean["opened_at"]), "clean.opened_at is not a datetime — run fix_types"
assert isinstance(clean["stage"].dtype, pd.CategoricalDtype), "clean.stage is not a category — run fix_types"
assert clean["margin_pct"].notna().all(), "clean.margin_pct has gaps — run handle_missing"
assert {"margin_pct_missing", "cost_outlier", "owner_key"} <= set(clean.columns), \
    "clean is missing a flag or owner_key — run every step in STEPS"
assert not {"owner_name", "owner_email", "contact_email"} & set(clean.columns), \
    "clean still has personal data — run pseudonymise"
assert isinstance(log, pd.DataFrame) and {"step", "rows"} <= set(log.columns), \
    "log must be a DataFrame with columns step and rows"
assert {f.__name__ for f in STEPS} <= set(log["step"]), "log needs one line per step (use step.__name__)"
assert log["rows"].iloc[-1] == 2_000, "the last line of log must show the 2000 rows of clean"
assert len(raw) == 2_040, "raw was changed — the pipeline must start from raw and never modify it"
passed(8)

✅ Exercise 8 passed


---

## Done

The same 2,000 bids as last week — but now you can say how good they are, and prove it.

| What you did | Where it shows up again |
|---|---|
| Measured before cleaning | The *before* column of every quality report you write |
| Declared types and a controlled vocabulary | Lesson 04 — group-bys on four stages, not fourteen |
| Chose which duplicate survives | Any table fed by more than one export |
| Filled *and* flagged | Lesson 05, where a model must not learn from guesses |
| Flagged instead of deleting | Every conversation with the owner of the source |
| Compared each value with its own group | Any metric that differs by sector, region or size |
| Measured distance in several dimensions at once | Lesson 05 features, lesson 11 drift monitoring |
| Pseudonymised and minimised | Your Assignment Brief's governance section |